# 05_04 Diagnosing bad Pseudo 890 origins

This notebook diagnoses the difficult Pseudo 890 weeks identified after aggregating the selected two-stage LightGBM forecasts in `05_03`. No model is retrained.

The weekly origins are seven days apart and the forecast horizon is seven days, so target windows do not overlap. Every target date belongs to exactly one origin: a bad origin is a bad calendar week, not forecast-age degradation within overlapping horizons.

## Diagnostic order

1. Rank weeks by absolute error kilograms before inspecting WAPE.
2. Use the article-day forecast/actual ratio to separate level from allocation error.
3. Locate the target days carrying each bad week's error.
4. Separate occurrence-probability from positive-quantity calibration.
5. Measure whether article errors are broad or concentrated.
6. Overlay public holidays, pre-holiday days, bridge days, school holidays, the first full week after Easter, and Corpus Christi as a non-public observance in Niedersachsen.
7. Check whether calendar demand patterns recur in the initial training origins and whether occurrence-stage errors recur in persisted validation predictions.

In [1]:
from pathlib import Path
import json
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.benchmark import load_benchmark_design
from src.models.lightgbm import LIGHTGBM_MODEL_LABELS, TWO_STAGE_MODEL_NAME
from src.models.lightgbm.features.builder import _holiday_calendar
from src.models.results import read_result, result_path

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)

ANALYSIS_MODEL = TWO_STAGE_MODEL_NAME
SOURCE_GROUP = 'Pseudo'
CATEGORY_ID = 890
BAD_ORIGIN_COUNT = 3
LEVEL_RATIO_TOLERANCE = 0.10

In [2]:
design = load_benchmark_design()
forecast_path = result_path(ANALYSIS_MODEL, design)
con = duckdb.connect()
con.execute('PRAGMA threads=4')
con.execute(
    '''
    CREATE OR REPLACE TEMP TABLE pseudo_890_rows AS
    SELECT
        ARTIKEL_ID::BIGINT AS ARTIKEL_ID,
        MARKT_ID::BIGINT AS MARKT_ID,
        CAST(origin AS DATE) AS origin,
        CAST(period AS DATE) AS period,
        actual::DOUBLE AS actual, forecast::DOUBLE AS forecast,
        occurrence_probability::DOUBLE AS occurrence_probability,
        positive_quantity_forecast::DOUBLE AS positive_quantity_forecast,
        is_active::BOOLEAN AS is_active
    FROM read_csv_auto(?)
    WHERE sourcing_group = ? AND category_id = ?
    ''',
    [str(forecast_path), SOURCE_GROUP, CATEGORY_ID],
)
con.execute(
    '''CREATE OR REPLACE TEMP VIEW pseudo_890_active AS
    SELECT * FROM pseudo_890_rows WHERE is_active'''
)

structure_audit = con.execute(
    '''
    WITH period_mapping AS (
        SELECT period, COUNT(DISTINCT origin) AS origins_per_target_date
        FROM pseudo_890_rows
        GROUP BY period
    ), origin_bounds AS (
        SELECT origin, MIN(period) AS first_period, MAX(period) AS last_period
        FROM pseudo_890_rows
        GROUP BY origin
    )
    SELECT
        (SELECT COUNT(DISTINCT origin) FROM pseudo_890_rows) AS origins,
        (SELECT COUNT(*) FROM period_mapping) AS target_dates,
        (SELECT MAX(origins_per_target_date) FROM period_mapping)
            AS maximum_origins_per_target_date,
        MIN(DATE_DIFF('day', first_period, last_period) + 1)
            AS minimum_horizon_days,
        MAX(DATE_DIFF('day', first_period, last_period) + 1)
            AS maximum_horizon_days
    FROM origin_bounds
    '''
).fetchdf()
origins = pd.DatetimeIndex(
    con.execute('SELECT DISTINCT origin FROM pseudo_890_rows ORDER BY origin')
    .fetchdf().origin
)
origin_gaps = origins.to_series().diff().dropna().dt.days.unique()
if not (
    structure_audit.loc[0, 'maximum_origins_per_target_date'] == 1
    and structure_audit.loc[0, 'minimum_horizon_days'] == design.forecast_horizon_days
    and structure_audit.loc[0, 'maximum_horizon_days'] == design.forecast_horizon_days
    and np.array_equal(origin_gaps, [design.origin_spacing_days])
):
    raise ValueError('Origins are not non-overlapping seven-day windows')
structure_audit['origin_spacing_days'] = int(origin_gaps[0])
print(f'Model: {LIGHTGBM_MODEL_LABELS[ANALYSIS_MODEL]}')
display(structure_audit)

Model: Two-stage LightGBM (occurrence × quantity)


,origins,target_dates,maximum_origins_per_target_date,minimum_horizon_days,maximum_horizon_days,origin_spacing_days
0,20,140,1,7,7,7


## 1–2. Real bad weeks, then level versus shape

Forecasts and actuals are first pooled across stores within each article-target-day. Origin WAPE is then computed from article-day errors. The three investigation weeks are selected by absolute error kilograms, not by WAPE. A forecast/actual ratio outside 0.90–1.10 is labelled a level miss; a high WAPE with a ratio near one is labelled an allocation/shape miss.

In [3]:
con.execute(
    '''
    CREATE OR REPLACE TEMP TABLE pseudo_890_article_day AS
    SELECT
        ARTIKEL_ID, origin, period,
        SUM(actual) AS actual, SUM(forecast) AS forecast
    FROM pseudo_890_active
    GROUP BY ARTIKEL_ID, origin, period
    '''
)
origin_results = con.execute(
    '''
    SELECT
        origin, SUM(actual) AS actual_kg, SUM(forecast) AS forecast_kg,
        SUM(ABS(forecast - actual)) AS absolute_error_kg,
        SUM(ABS(forecast - actual)) / NULLIF(SUM(actual), 0) AS pooled_wape,
        SUM(forecast) / NULLIF(SUM(actual), 0) AS ratio
    FROM pseudo_890_article_day
    GROUP BY origin
    ORDER BY origin
    '''
).fetchdf()
origin_results['absolute_error_rank'] = (
    origin_results.absolute_error_kg.rank(method='min', ascending=False).astype(int)
)
origin_results['wape_rank'] = (
    origin_results.pooled_wape.rank(method='min', ascending=False).astype(int)
)
origin_results['actual_volume_rank'] = (
    origin_results.actual_kg.rank(method='min', ascending=False).astype(int)
)
origin_results['diagnosis'] = np.where(
    origin_results.ratio.sub(1).abs().gt(LEVEL_RATIO_TOLERANCE),
    np.where(origin_results.ratio.gt(1), 'level: overforecast', 'level: underforecast'),
    'allocation/shape',
)
origin_results = origin_results.sort_values(
    'absolute_error_rank'
).reset_index(drop=True)
bad_origins = origin_results.head(BAD_ORIGIN_COUNT).copy()
bad_origin_keys = bad_origins[['origin']].copy()
con.register('bad_origin_keys', bad_origin_keys)

display(origin_results.style.format({
    'origin': '{:%Y-%m-%d}', 'actual_kg': '{:,.1f}',
    'forecast_kg': '{:,.1f}', 'absolute_error_kg': '{:,.1f}',
    'pooled_wape': '{:.2%}', 'ratio': '{:.3f}',
    'absolute_error_rank': '{:,.0f}', 'wape_rank': '{:,.0f}',
    'actual_volume_rank': '{:,.0f}',
}))
print('Selected bad origins (top absolute-error weeks)')
display(bad_origins.style.format({
    'origin': '{:%Y-%m-%d}', 'actual_kg': '{:,.1f}',
    'forecast_kg': '{:,.1f}', 'absolute_error_kg': '{:,.1f}',
    'pooled_wape': '{:.2%}', 'ratio': '{:.3f}',
}))

,origin,actual_kg,forecast_kg,absolute_error_kg,pooled_wape,ratio,absolute_error_rank,wape_rank,actual_volume_rank,diagnosis
0,2026-05-18,"177,175.4","136,755.1","63,302.5",35.73%,0.772,1,3,1,level: underforecast
1,2026-04-27,"145,644.4","112,578.7","50,783.7",34.87%,0.773,2,4,3,level: underforecast
2,2026-04-13,"119,736.0","160,672.5","44,932.9",37.53%,1.342,3,1,10,level: overforecast
3,2026-03-30,"171,458.9","138,942.1","44,131.6",25.74%,0.810,4,7,2,level: underforecast
4,2026-06-22,"101,791.4","102,281.2","38,026.6",37.36%,1.005,5,2,20,allocation/shape
5,2026-06-15,"130,869.0","137,590.3","34,067.3",26.03%,1.051,6,6,5,allocation/shape
6,2026-05-11,"111,623.3","117,579.3","32,457.6",29.08%,1.053,7,5,15,allocation/shape
7,2026-06-08,"123,124.0","138,683.7","31,027.1",25.20%,1.126,8,9,9,level: overforecast
8,2026-05-04,"126,240.2","115,108.2","28,455.2",22.54%,0.912,9,11,6,allocation/shape
9,2026-03-02,"141,624.4","140,222.6","28,303.1",19.98%,0.990,10,17,4,allocation/shape


Selected bad origins (top absolute-error weeks)


,origin,actual_kg,forecast_kg,absolute_error_kg,pooled_wape,ratio,absolute_error_rank,wape_rank,actual_volume_rank,diagnosis
0,2026-05-18,"177,175.4","136,755.1","63,302.5",35.73%,0.772,1,3,1,level: underforecast
1,2026-04-27,"145,644.4","112,578.7","50,783.7",34.87%,0.773,2,4,3,level: underforecast
2,2026-04-13,"119,736.0","160,672.5","44,932.9",37.53%,1.342,3,1,10,level: overforecast


## Calendar overlay

Public holidays and ±3-day event names reproduce the Niedersachsen calendar used by the model features. School-holiday ranges come from the [official Niedersachsen Ministry of Education schedule](https://www.mk.niedersachsen.de/startseite/service/ferientermine/schulferien-6491.html). A bridge day is a Monday before a Tuesday public holiday or a Friday after a Thursday public holiday. `week_after_easter` is the first complete Monday–Sunday week following Easter Monday. Corpus Christi is tagged as an observance on 4 June 2026 but is not treated as a statewide public holiday in Niedersachsen.

In [4]:
SCHOOL_HOLIDAY_RANGES = [
    ('Easter school holiday', '2025-04-07', '2025-04-19'),
    ('Ascension school holiday', '2025-05-30', '2025-05-30'),
    ('Pentecost school holiday', '2025-06-10', '2025-06-10'),
    ('Summer school holiday', '2025-07-03', '2025-08-13'),
    ('Autumn school holiday', '2025-10-13', '2025-10-25'),
    ('Christmas school holiday', '2025-12-22', '2026-01-05'),
    ('Half-year school holiday', '2026-02-02', '2026-02-03'),
    ('Easter school holiday', '2026-03-23', '2026-04-07'),
    ('Ascension school holiday', '2026-05-15', '2026-05-15'),
    ('Pentecost school holiday', '2026-05-26', '2026-05-26'),
    ('Summer school holiday', '2026-07-02', '2026-08-12'),
]
EASTER_MONDAYS = [pd.Timestamp('2025-04-21'), pd.Timestamp('2026-04-06')]

def make_calendar_tags(start, end):
    calendar = _holiday_calendar(start, end).copy()
    calendar['period'] = pd.to_datetime(calendar.period)
    calendar['weekday'] = calendar.period.dt.day_name()
    holiday_dates = set(calendar.loc[calendar.is_public_holiday, 'period'])
    calendar['day_before_public_holiday'] = calendar.period.add(
        pd.Timedelta(days=1)
    ).isin(holiday_dates)
    calendar['bridge_day'] = (
        (calendar.period.dt.weekday.eq(4)
         & calendar.period.sub(pd.Timedelta(days=1)).isin(holiday_dates))
        | (calendar.period.dt.weekday.eq(0)
           & calendar.period.add(pd.Timedelta(days=1)).isin(holiday_dates))
    )
    calendar['school_holiday_name'] = 'none'
    for name, first_day, last_day in SCHOOL_HOLIDAY_RANGES:
        in_range = calendar.period.between(first_day, last_day)
        calendar.loc[in_range, 'school_holiday_name'] = name
    calendar['school_holiday'] = calendar.school_holiday_name.ne('none')
    calendar['week_after_easter'] = False
    for easter_monday in EASTER_MONDAYS:
        calendar.loc[calendar.period.between(
            easter_monday + pd.Timedelta(days=7),
            easter_monday + pd.Timedelta(days=13),
        ), 'week_after_easter'] = True
    calendar['special_observance'] = np.where(
        calendar.period.eq(pd.Timestamp('2026-06-04')),
        'Corpus Christi (not NI public holiday)', 'none',
    )
    calendar['calendar_context'] = np.select(
        [
            calendar.is_public_holiday,
            calendar.day_before_public_holiday,
            calendar.bridge_day,
            calendar.week_after_easter,
            calendar.school_holiday,
            calendar.special_observance.ne('none'),
        ],
        [
            'public holiday', 'day before public holiday', 'bridge day',
            'week after Easter', 'school holiday', 'special observance',
        ],
        default='ordinary',
    )
    return calendar

training_summary = read_result(
    ANALYSIS_MODEL, design, artifact='training_summaries'
)
first_refit = training_summary.sort_values('evaluation_origin').iloc[0]
calendar_tags = make_calendar_tags(
    pd.Timestamp(first_refit.training_start),
    origins.max() + pd.Timedelta(days=design.forecast_horizon_days - 1),
)
con.register('calendar_tags', calendar_tags)
display(calendar_tags.loc[
    calendar_tags.calendar_context.ne('ordinary')
    & calendar_tags.period.between(origins.min(), origins.max() + pd.Timedelta(days=6)),
    [
        'period', 'weekday', 'calendar_context', 'event_name',
        'holiday_event_window', 'school_holiday_name', 'special_observance',
    ],
].reset_index(drop=True))

,period,weekday,calendar_context,event_name,holiday_event_window,school_holiday_name,special_observance
0,2026-03-23,Monday,school holiday,none,none,Easter school holiday,none
1,2026-03-24,Tuesday,school holiday,none,none,Easter school holiday,none
2,2026-03-25,Wednesday,school holiday,none,none,Easter school holiday,none
3,2026-03-26,Thursday,school holiday,none,none,Easter school holiday,none
4,2026-03-27,Friday,school holiday,none,none,Easter school holiday,none
5,2026-03-28,Saturday,school holiday,none,none,Easter school holiday,none
6,2026-03-29,Sunday,school holiday,none,none,Easter school holiday,none
7,2026-03-30,Monday,school holiday,none,none,Easter school holiday,none
8,2026-03-31,Tuesday,school holiday,Karfreitag,before_holiday_1_3d,Easter school holiday,none
9,2026-04-01,Wednesday,school holiday,Karfreitag,before_holiday_1_3d,Easter school holiday,none


## 3. Which target day carries the error?

`article_day_absolute_error_kg` retains article-level errors after pooling stores. `portfolio_level_error_kg` is the absolute error after also pooling articles and therefore isolates the daily total-level miss. Public-holiday dates with no active forecast rows remain visible as zero-volume calendar rows.

In [5]:
bad_day_metrics = con.execute(
    '''
    SELECT
        d.origin, d.period, COUNT(*) AS articles,
        SUM(d.actual) AS actual_kg, SUM(d.forecast) AS forecast_kg,
        SUM(ABS(d.forecast - d.actual)) AS article_day_absolute_error_kg,
        ABS(SUM(d.forecast) - SUM(d.actual)) AS portfolio_level_error_kg
    FROM pseudo_890_article_day AS d
    INNER JOIN bad_origin_keys AS b USING (origin)
    GROUP BY d.origin, d.period
    '''
).fetchdf()
bad_day_grid = pd.concat([
    pd.DataFrame({
        'origin': origin,
        'period': pd.date_range(origin, periods=design.forecast_horizon_days),
    })
    for origin in bad_origins.origin
], ignore_index=True)
bad_days = (
    bad_day_grid.merge(
        bad_day_metrics, on=['origin', 'period'], how='left', validate='one_to_one'
    )
    .merge(calendar_tags, on='period', how='left', validate='many_to_one')
)
metric_columns = [
    'articles', 'actual_kg', 'forecast_kg',
    'article_day_absolute_error_kg', 'portfolio_level_error_kg',
]
bad_days[metric_columns] = bad_days[metric_columns].fillna(0)
bad_days['ratio'] = bad_days.forecast_kg / bad_days.actual_kg.replace(0, np.nan)
bad_days['error_share_within_origin'] = (
    bad_days.article_day_absolute_error_kg
    / bad_days.groupby('origin').article_day_absolute_error_kg.transform('sum')
)
bad_days = bad_days.sort_values(['origin', 'period']).reset_index(drop=True)
display(bad_days[[
    'origin', 'period', 'weekday', 'calendar_context', 'event_name',
    'holiday_event_window', 'school_holiday_name', 'articles',
    'actual_kg', 'forecast_kg', 'ratio',
    'article_day_absolute_error_kg', 'portfolio_level_error_kg',
    'error_share_within_origin',
]].style.format({
    'origin': '{:%Y-%m-%d}', 'period': '{:%Y-%m-%d}',
    'articles': '{:,.0f}', 'actual_kg': '{:,.1f}',
    'forecast_kg': '{:,.1f}', 'ratio': '{:.3f}',
    'article_day_absolute_error_kg': '{:,.1f}',
    'portfolio_level_error_kg': '{:,.1f}',
    'error_share_within_origin': '{:.1%}',
}))

contributing_stores = con.execute(
    '''
    SELECT
        r.origin, r.period, COUNT(DISTINCT r.MARKT_ID) AS open_stores,
        LIST_SORT(LIST(DISTINCT r.MARKT_ID)) AS open_store_ids,
        SUM(r.actual) AS actual_kg, SUM(r.forecast) AS forecast_kg
    FROM pseudo_890_active AS r
    INNER JOIN bad_origin_keys AS b USING (origin)
    GROUP BY r.origin, r.period
    '''
).fetchdf()
stores_by_day = (
    bad_day_grid.merge(
        contributing_stores,
        on=['origin', 'period'], how='left', validate='one_to_one',
    )
    .merge(
        calendar_tags[['period', 'weekday']],
        on='period', how='left', validate='many_to_one',
    )
)
stores_by_day['open_stores'] = stores_by_day.open_stores.fillna(0).astype(int)
stores_by_day[['actual_kg', 'forecast_kg']] = stores_by_day[[
    'actual_kg', 'forecast_kg'
]].fillna(0)
stores_by_day['open_store_key'] = stores_by_day.open_store_ids.apply(
    lambda values: tuple(int(value) for value in values)
    if isinstance(values, np.ndarray) else tuple()
)

store_pool_audit = stores_by_day.merge(
    bad_days[['origin', 'period', 'actual_kg', 'forecast_kg']],
    on=['origin', 'period'], suffixes=('_stores', '_day_table'),
    validate='one_to_one',
)
if not (
    np.allclose(store_pool_audit.actual_kg_stores, store_pool_audit.actual_kg_day_table)
    and np.allclose(
        store_pool_audit.forecast_kg_stores, store_pool_audit.forecast_kg_day_table
    )
):
    raise ValueError('Store-level pools do not reproduce the Section 3 totals')
sunday_stores = stores_by_day.loc[stores_by_day.weekday.eq('Sunday')]
if not sunday_stores.open_stores.eq(2).all():
    raise ValueError('Every Sunday must contain exactly two open stores')

unique_store_keys = list(dict.fromkeys(stores_by_day.open_store_key))
store_set_labels = {key: f'S{position}' for position, key in enumerate(unique_store_keys)}
stores_by_day['store_set'] = stores_by_day.open_store_key.apply(
    lambda key: store_set_labels[key]
)
store_set_details = pd.DataFrame([
    {
        'store_set': store_set_labels[key],
        'n_open_stores': len(key),
        'open_store_ids': ', '.join(str(value) for value in key) or 'none',
    }
    for key in unique_store_keys
])

print('Stores contributing to each pooled day')
display(stores_by_day[[
    'origin', 'period', 'weekday', 'open_stores', 'store_set',
    'actual_kg', 'forecast_kg',
]].style.format({
    'origin': '{:%Y-%m-%d}', 'period': '{:%Y-%m-%d}',
    'open_stores': '{:,.0f}', 'actual_kg': '{:,.1f}',
    'forecast_kg': '{:,.1f}',
}))
print('Store-set definitions (IDs listed once per unique set)')
display(store_set_details)

,origin,period,weekday,calendar_context,event_name,holiday_event_window,school_holiday_name,articles,actual_kg,forecast_kg,ratio,article_day_absolute_error_kg,portfolio_level_error_kg,error_share_within_origin
0,2026-04-13,2026-04-13,Monday,week after Easter,none,none,none,278,"15,569.8","21,354.1",1.372,"6,451.8","5,784.3",14.4%
1,2026-04-13,2026-04-14,Tuesday,week after Easter,none,none,none,278,"14,218.3","20,323.7",1.429,"6,334.5","6,105.5",14.1%
2,2026-04-13,2026-04-15,Wednesday,week after Easter,none,none,none,278,"14,566.3","20,542.9",1.410,"6,231.4","5,976.6",13.9%
3,2026-04-13,2026-04-16,Thursday,week after Easter,none,none,none,278,"18,096.7","25,340.0",1.400,"7,514.2","7,243.3",16.7%
4,2026-04-13,2026-04-17,Friday,week after Easter,none,none,none,278,"26,457.0","32,369.0",1.223,"7,945.0","5,912.1",17.7%
5,2026-04-13,2026-04-18,Saturday,week after Easter,none,none,none,278,"30,771.0","40,636.3",1.321,"10,387.3","9,865.3",23.1%
6,2026-04-13,2026-04-19,Sunday,week after Easter,none,none,none,118,57.0,106.4,1.867,68.8,49.4,0.2%
7,2026-04-27,2026-04-27,Monday,ordinary,none,none,none,278,"13,731.7","14,085.4",1.026,"1,961.2",353.7,3.9%
8,2026-04-27,2026-04-28,Tuesday,ordinary,Erster Mai,before_holiday_1_3d,none,278,"14,813.1","16,927.0",1.143,"3,300.5","2,113.9",6.5%
9,2026-04-27,2026-04-29,Wednesday,ordinary,Erster Mai,before_holiday_1_3d,none,278,"25,514.5","25,106.0",0.984,"5,307.7",408.6,10.5%


Stores contributing to each pooled day


,origin,period,weekday,open_stores,store_set,actual_kg,forecast_kg
0,2026-05-18,2026-05-18,Monday,193,S0,"19,349.8","17,679.8"
1,2026-05-18,2026-05-19,Tuesday,193,S0,"17,334.3","17,165.9"
2,2026-05-18,2026-05-20,Wednesday,193,S0,"19,468.8","18,019.4"
3,2026-05-18,2026-05-21,Thursday,193,S0,"25,187.3","21,426.8"
4,2026-05-18,2026-05-22,Friday,193,S0,"43,484.0","29,064.0"
5,2026-05-18,2026-05-23,Saturday,193,S0,"52,351.2","33,335.4"
6,2026-05-18,2026-05-24,Sunday,2,S1,0.0,63.8
7,2026-04-27,2026-04-27,Monday,193,S0,"13,731.7","14,085.4"
8,2026-04-27,2026-04-28,Tuesday,193,S0,"14,813.1","16,927.0"
9,2026-04-27,2026-04-29,Wednesday,193,S0,"25,514.5","25,106.0"


Store-set definitions (IDs listed once per unique set)


,store_set,n_open_stores,open_store_ids
0,S0,193,"1100001, 1100002, 1100003, 1100004, 1100005, 1..."
1,S1,2,"1100079, 1100084"
2,S2,0,none


## 4. Which two-stage component missed?

Occurrence calibration compares the observed positive-row rate with mean predicted probability across all active rows. Positive-quantity calibration compares actual quantity with the predicted conditional quantity only on rows where demand was positive, matching the quantity-stage target used in `05_01`.

In [6]:
stage_by_origin = con.execute(
    '''
    SELECT
        origin, COUNT(*) AS active_rows,
        AVG((actual > 0)::INTEGER) AS actual_occurrence_rate,
        AVG(occurrence_probability) AS predicted_occurrence_rate,
        AVG(actual) FILTER (WHERE actual > 0) AS actual_positive_mean,
        AVG(positive_quantity_forecast) FILTER (WHERE actual > 0)
            AS predicted_positive_mean,
        AVG(actual) AS actual_mean_per_row, AVG(forecast) AS forecast_mean_per_row
    FROM pseudo_890_active
    GROUP BY origin
    ORDER BY origin
    '''
).fetchdf()
stage_by_origin['occurrence_gap_pp'] = 100 * (
    stage_by_origin.predicted_occurrence_rate
    - stage_by_origin.actual_occurrence_rate
)
stage_by_origin['positive_mean_ratio'] = (
    stage_by_origin.predicted_positive_mean / stage_by_origin.actual_positive_mean
)
bad_stage = stage_by_origin.merge(
    bad_origin_keys, on='origin', how='inner', validate='one_to_one'
).sort_values('origin')
display(bad_stage.style.format({
    'origin': '{:%Y-%m-%d}', 'active_rows': '{:,.0f}',
    'actual_occurrence_rate': '{:.2%}',
    'predicted_occurrence_rate': '{:.2%}',
    'occurrence_gap_pp': '{:+.2f}',
    'actual_positive_mean': '{:.3f}', 'predicted_positive_mean': '{:.3f}',
    'positive_mean_ratio': '{:.3f}',
    'actual_mean_per_row': '{:.3f}', 'forecast_mean_per_row': '{:.3f}',
}))

,origin,active_rows,actual_occurrence_rate,predicted_occurrence_rate,actual_positive_mean,predicted_positive_mean,actual_mean_per_row,forecast_mean_per_row,occurrence_gap_pp,positive_mean_ratio
0,2026-04-13,"111,704",38.32%,41.78%,2.797,3.637,1.072,1.438,+3.46,1.300
1,2026-04-27,"93,149",42.11%,40.09%,3.713,2.953,1.564,1.209,-2.03,0.795
2,2026-05-18,"111,776",40.99%,40.10%,3.867,2.993,1.585,1.223,-0.89,0.774


## 5. Broad or article-concentrated?

Article contributions sum absolute article-day errors across the seven-day origin. The top-10 share distinguishes a small product event from a broadly distributed calendar or level miss.

In [7]:
article_contributions = con.execute(
    '''
    WITH article_errors AS (
        SELECT
            d.origin, d.ARTIKEL_ID, SUM(d.actual) AS actual_kg,
            SUM(d.forecast) AS forecast_kg,
            SUM(ABS(d.forecast - d.actual)) AS absolute_error_kg
        FROM pseudo_890_article_day AS d
        INNER JOIN bad_origin_keys AS b USING (origin)
        GROUP BY d.origin, d.ARTIKEL_ID
    )
    SELECT
        *, ROW_NUMBER() OVER (
            PARTITION BY origin ORDER BY absolute_error_kg DESC
        ) AS error_rank,
        absolute_error_kg / SUM(absolute_error_kg) OVER (PARTITION BY origin)
            AS error_share
    FROM article_errors
    ORDER BY origin, error_rank
    '''
).fetchdf()
concentration_summary = (
    article_contributions.groupby('origin')
    .agg(
        articles=('ARTIKEL_ID', 'nunique'),
        top_1_error_share=('error_share', 'first'),
        top_10_error_share=(
            'error_share', lambda values: values.iloc[:10].sum()
        ),
    )
    .reset_index()
)
display(concentration_summary.style.format({
    'origin': '{:%Y-%m-%d}', 'articles': '{:,.0f}',
    'top_1_error_share': '{:.1%}', 'top_10_error_share': '{:.1%}',
}))
display(article_contributions.loc[
    article_contributions.error_rank.le(10)
].style.format({
    'origin': '{:%Y-%m-%d}', 'ARTIKEL_ID': '{:,.0f}',
    'actual_kg': '{:,.1f}', 'forecast_kg': '{:,.1f}',
    'absolute_error_kg': '{:,.1f}', 'error_rank': '{:,.0f}',
    'error_share': '{:.1%}',
}))

,origin,articles,top_1_error_share,top_10_error_share
0,2026-04-13,278,15.3%,46.6%
1,2026-04-27,278,11.1%,59.5%
2,2026-05-18,278,14.4%,55.9%


,origin,ARTIKEL_ID,actual_kg,forecast_kg,absolute_error_kg,error_rank,error_share
0,2026-04-13,"317,123","6,695.0","13,588.2","6,893.2",1,15.3%
1,2026-04-13,"316,623","9,310.5","11,578.3","2,267.8",2,5.0%
2,2026-04-13,"316,968","1,765.4","3,876.7","2,111.3",3,4.7%
3,2026-04-13,"329,473",820.3,"2,596.2","1,775.8",4,4.0%
4,2026-04-13,"317,099","2,422.8","4,198.4","1,775.6",5,4.0%
5,2026-04-13,"369,085","1,914.3","3,411.5","1,497.3",6,3.3%
6,2026-04-13,"317,161","2,602.7","3,873.2","1,270.6",7,2.8%
7,2026-04-13,"316,821","2,710.7","3,941.3","1,230.5",8,2.7%
8,2026-04-13,"317,207","1,949.9","3,078.7","1,128.9",9,2.5%
9,2026-04-13,"317,157","2,349.7","3,084.9","1,004.3",10,2.2%


## 6. Calendar contexts across all evaluation weeks

This table prevents calendar interpretation from relying only on the three selected weeks. It aggregates article-day errors over every evaluation date in the same context. Contexts are mutually exclusive using the priority documented above; the detailed bad-day table retains the underlying event fields.

In [8]:
evaluation_date_metrics = con.execute(
    '''
    SELECT
        period, SUM(actual) AS actual_kg, SUM(forecast) AS forecast_kg,
        SUM(ABS(forecast - actual)) AS absolute_error_kg
    FROM pseudo_890_article_day
    GROUP BY period
    '''
).fetchdf().merge(
    calendar_tags[['period', 'calendar_context']],
    on='period', how='left', validate='one_to_one',
)
evaluation_calendar_summary = (
    evaluation_date_metrics.groupby('calendar_context', observed=True)
    .agg(
        target_dates=('period', 'nunique'),
        actual_kg=('actual_kg', 'sum'),
        forecast_kg=('forecast_kg', 'sum'),
        absolute_error_kg=('absolute_error_kg', 'sum'),
    )
    .reset_index()
)
evaluation_calendar_summary['pooled_wape'] = (
    evaluation_calendar_summary.absolute_error_kg
    / evaluation_calendar_summary.actual_kg
)
evaluation_calendar_summary['ratio'] = (
    evaluation_calendar_summary.forecast_kg
    / evaluation_calendar_summary.actual_kg
)
display(evaluation_calendar_summary.sort_values(
    'absolute_error_kg', ascending=False
).style.format({
    'target_dates': '{:,.0f}', 'actual_kg': '{:,.1f}',
    'forecast_kg': '{:,.1f}', 'absolute_error_kg': '{:,.1f}',
    'pooled_wape': '{:.2%}', 'ratio': '{:.3f}',
}))

,calendar_context,target_dates,actual_kg,forecast_kg,absolute_error_kg,pooled_wape,ratio
2,ordinary,90,"1,625,781.6","1,627,088.9","410,287.8",25.24%,1.001
3,school holiday,31,"577,219.7","545,167.9","128,876.9",22.33%,0.944
1,day before public holiday,5,"129,957.2","89,642.7","46,034.1",35.42%,0.690
5,week after Easter,7,"119,736.0","160,672.5","44,932.9",37.53%,1.342
0,bridge day,1,"24,181.3","25,340.7","5,557.7",22.98%,1.048
4,special observance,1,"12,551.1","15,674.4","4,552.7",36.27%,1.249


## 7. Does the calendar pattern generalize?

The initial 48 fit origins provide non-overlapping historical target weeks from March 2025 through January 2026. They can confirm whether demand occurrence and positive quantities change repeatedly in the same calendar contexts, although they cannot supply forecast errors for a model that was not persisted on its own fit rows. Persisted rolling validation predictions add an out-of-sample check for the occurrence stage.

In [9]:
FEATURE_STORE_ROOT = ROOT / 'data' / 'processed' / 'model_features'

def matching_feature_generation():
    matches = []
    for manifest_path in FEATURE_STORE_ROOT.rglob('manifest.json'):
        try:
            manifest = json.loads(manifest_path.read_text())
            cached_design = manifest['signature']['forecast_design']
        except (OSError, KeyError, json.JSONDecodeError, TypeError):
            continue
        same_design = (
            cached_design.get('first_origin') == design.first_origin.date().isoformat()
            and int(cached_design.get('forecast_horizon_days', -1))
                == design.forecast_horizon_days
            and int(cached_design.get('origin_spacing_days', -1))
                == design.origin_spacing_days
            and Path(cached_design.get('data_dir', '')).resolve()
                == design.data_dir.resolve()
        )
        if same_design:
            matches.append((manifest_path.stat().st_mtime, manifest_path.parent))
    if not matches:
        raise FileNotFoundError('No matching feature-cache generation found')
    return max(matches, key=lambda item: item[0])[1]

feature_generation = matching_feature_generation()
fit_origins = pd.date_range(
    first_refit.training_start, first_refit.training_end,
    freq=f'{design.origin_spacing_days}D',
)
fit_paths = [
    feature_generation / f'origin={origin.date().isoformat()}' / 'features.parquet'
    for origin in fit_origins
]
missing_fit_paths = [path for path in fit_paths if not path.exists()]
if missing_fit_paths:
    raise FileNotFoundError(f'Missing training feature partitions: {missing_fit_paths[:3]}')
con.execute(
    '''
    CREATE OR REPLACE TEMP TABLE pseudo_890_training_actuals AS
    SELECT
        CAST(origin AS DATE) AS origin, CAST(period AS DATE) AS period,
        actual::DOUBLE AS actual
    FROM read_parquet(?, hive_partitioning=false)
    WHERE is_active AND sourcing_group = ? AND category_id = ?
    ''',
    [[str(path) for path in fit_paths], SOURCE_GROUP, CATEGORY_ID],
)
print(
    f'Initial fit origins: {len(fit_origins)} '
    f'({fit_origins.min().date()} to {fit_origins.max().date()})'
)

Initial fit origins: 48 (2025-03-03 to 2026-01-26)


In [10]:
historical_context = con.execute(
    '''
    WITH samples AS (
        SELECT 'initial fit origins' AS sample, period, actual
        FROM pseudo_890_training_actuals
        UNION ALL
        SELECT 'evaluation origins' AS sample, period, actual
        FROM pseudo_890_active
    )
    SELECT
        s.sample, c.calendar_context, COUNT(*) AS active_rows,
        COUNT(DISTINCT s.period) AS target_dates,
        AVG(s.actual) AS mean_actual_kg,
        AVG((s.actual > 0)::INTEGER) AS occurrence_rate,
        AVG(s.actual) FILTER (WHERE s.actual > 0) AS positive_quantity_mean
    FROM samples AS s
    INNER JOIN calendar_tags AS c USING (period)
    GROUP BY s.sample, c.calendar_context
    ORDER BY c.calendar_context, s.sample
    '''
).fetchdf()
display(historical_context.style.format({
    'active_rows': '{:,.0f}', 'target_dates': '{:,.0f}',
    'mean_actual_kg': '{:.3f}', 'occurrence_rate': '{:.2%}',
    'positive_quantity_mean': '{:.3f}',
}))

validation_path = result_path(
    ANALYSIS_MODEL, design, artifact='validation_predictions'
)
validation_occurrence = con.execute(
    '''
    SELECT
        c.calendar_context, COUNT(*) AS active_rows,
        COUNT(DISTINCT CAST(v.period AS DATE)) AS target_dates,
        AVG((v.actual > 0)::INTEGER) AS actual_occurrence_rate,
        AVG(v.occurrence_probability) AS predicted_occurrence_rate
    FROM read_csv_auto(?) AS v
    INNER JOIN calendar_tags AS c
        ON CAST(v.period AS DATE) = c.period
    WHERE v.is_active AND v.sourcing_group = ? AND v.category_id = ?
    GROUP BY c.calendar_context
    ORDER BY c.calendar_context
    ''',
    [str(validation_path), SOURCE_GROUP, CATEGORY_ID],
).fetchdf()
validation_occurrence['occurrence_gap_pp'] = 100 * (
    validation_occurrence.predicted_occurrence_rate
    - validation_occurrence.actual_occurrence_rate
)
print('Persisted rolling-validation occurrence calibration')
display(validation_occurrence.style.format({
    'active_rows': '{:,.0f}', 'target_dates': '{:,.0f}',
    'actual_occurrence_rate': '{:.2%}',
    'predicted_occurrence_rate': '{:.2%}',
    'occurrence_gap_pp': '{:+.2f}',
}))

,sample,calendar_context,active_rows,target_dates,mean_actual_kg,occurrence_rate,positive_quantity_mean
0,evaluation origins,bridge day,"18,594",1,1.300,43.35%,3.000
1,initial fit origins,bridge day,"54,246",3,1.540,42.27%,3.643
2,evaluation origins,day before public holiday,"56,152",5,2.314,48.56%,4.766
3,initial fit origins,day before public holiday,"127,412",9,1.968,43.42%,4.533
4,evaluation origins,ordinary,"1,434,144",90,1.134,37.51%,3.022
5,initial fit origins,ordinary,"3,617,963",234,1.243,37.01%,3.359
6,evaluation origins,school holiday,"502,999",31,1.148,37.37%,3.070
7,initial fit origins,school holiday,"1,197,581",76,1.278,38.84%,3.291
8,evaluation origins,special observance,"18,604",1,0.675,27.33%,2.469
9,evaluation origins,week after Easter,"111,704",7,1.072,38.32%,2.797


Persisted rolling-validation occurrence calibration


,calendar_context,active_rows,target_dates,actual_occurrence_rate,predicted_occurrence_rate,occurrence_gap_pp
0,bridge day,"18,594",1,43.35%,41.69%,-1.66
1,day before public holiday,"56,152",5,48.56%,44.81%,-3.75
2,ordinary,"1,674,416",106,37.09%,37.32%,+0.23
3,school holiday,"260,009",15,36.17%,35.85%,-0.32
4,special observance,"18,604",1,27.33%,35.05%,+7.73
5,week after Easter,"111,704",7,38.32%,41.80%,+3.47


In [11]:
bad_origin_text = '; '.join(
    f"{row.origin:%Y-%m-%d}: {row.diagnosis}, ratio {row.ratio:.3f}"
    for row in bad_origins.itertuples()
)
top10_text = '; '.join(
    f"{row.origin:%Y-%m-%d}: {row.top_10_error_share:.1%}"
    for row in concentration_summary.itertuples()
)

def stage_value(origin, column):
    return bad_stage.loc[bad_stage.origin.eq(pd.Timestamp(origin)), column].iloc[0]

april_13_occurrence_gap = stage_value('2026-04-13', 'occurrence_gap_pp')
april_13_quantity_ratio = stage_value('2026-04-13', 'positive_mean_ratio')
april_27_occurrence_gap = stage_value('2026-04-27', 'occurrence_gap_pp')
april_27_quantity_ratio = stage_value('2026-04-27', 'positive_mean_ratio')
may_18_occurrence_gap = stage_value('2026-05-18', 'occurrence_gap_pp')
may_18_quantity_ratio = stage_value('2026-05-18', 'positive_mean_ratio')
april_30_error_share = bad_days.loc[
    bad_days.period.eq(pd.Timestamp('2026-04-30')),
    'error_share_within_origin',
].iloc[0]
may_22_23_error_share = bad_days.loc[
    bad_days.period.isin(pd.to_datetime(['2026-05-22', '2026-05-23'])),
    'error_share_within_origin',
].sum()
validation_day_before_gap = validation_occurrence.loc[
    validation_occurrence.calendar_context.eq('day before public holiday'),
    'occurrence_gap_pp',
].iloc[0]
validation_day_before_underprediction = -validation_day_before_gap
validation_easter_gap = validation_occurrence.loc[
    validation_occurrence.calendar_context.eq('week after Easter'),
    'occurrence_gap_pp',
].iloc[0]
display(Markdown(f'''
## Interpretation

The three largest article-day absolute-error weeks are genuine bad weeks, not
small-denominator WAPE artifacts. Their level diagnoses are: **{bad_origin_text}**.

The mechanisms differ by week:

- **13 April:** Overforecasting persists across the full week after Easter. The
  occurrence stage is high by **{april_13_occurrence_gap:+.2f} points** and the
  conditional-quantity ratio is **{april_13_quantity_ratio:.3f}**. Rolling
  validation repeats a **{validation_easter_gap:+.2f}-point** occurrence gap, but
  historical fit rows have higher—not lower—week-after-Easter demand than the
  evaluation sample. The low 2026 week is therefore not a recurring low-demand
  calendar effect.
- **27 April:** The day before 1 May alone carries
  **{april_30_error_share:.1%}** of the week's error. Occurrence is only
  **{april_27_occurrence_gap:+.2f} points** off, while predicted positive quantity
  is only **{april_27_quantity_ratio:.1%}** of actual. This is mainly a
  pre-holiday basket-size miss.
- **18 May:** Friday and Saturday before Pentecost carry
  **{may_22_23_error_share:.1%}** of the week's error. Occurrence is close
  (**{may_18_occurrence_gap:+.2f} points**), but predicted positive quantity is
  only **{may_18_quantity_ratio:.1%}** of actual. This is again a quantity-stage
  pre-holiday demand spike.

Top-10 article shares are **{top10_text}**. The two underforecast weeks are more
article-concentrated than 13 April, but none is a single-product anomaly. The
historical fit rows show elevated demand before public holidays, and rolling
validation underpredicts their occurrence by
**{validation_day_before_underprediction:.2f} percentage points**. That repeated
pre-holiday pattern is the strongest candidate for future
feature or model work; one-off calendar effects should remain descriptive.
'''))


## Interpretation

The three largest article-day absolute-error weeks are genuine bad weeks, not
small-denominator WAPE artifacts. Their level diagnoses are: **2026-05-18: level: underforecast, ratio 0.772; 2026-04-27: level: underforecast, ratio 0.773; 2026-04-13: level: overforecast, ratio 1.342**.

The mechanisms differ by week:

- **13 April:** Overforecasting persists across the full week after Easter. The
  occurrence stage is high by **+3.46 points** and the
  conditional-quantity ratio is **1.300**. Rolling
  validation repeats a **+3.47-point** occurrence gap, but
  historical fit rows have higher—not lower—week-after-Easter demand than the
  evaluation sample. The low 2026 week is therefore not a recurring low-demand
  calendar effect.
- **27 April:** The day before 1 May alone carries
  **54.7%** of the week's error. Occurrence is only
  **-2.03 points** off, while predicted positive quantity
  is only **79.5%** of actual. This is mainly a
  pre-holiday basket-size miss.
- **18 May:** Friday and Saturday before Pentecost carry
  **73.0%** of the week's error. Occurrence is close
  (**-0.89 points**), but predicted positive quantity is
  only **77.4%** of actual. This is again a quantity-stage
  pre-holiday demand spike.

Top-10 article shares are **2026-04-13: 46.6%; 2026-04-27: 59.5%; 2026-05-18: 55.9%**. The two underforecast weeks are more
article-concentrated than 13 April, but none is a single-product anomaly. The
historical fit rows show elevated demand before public holidays, and rolling
validation underpredicts their occurrence by
**3.75 percentage points**. That repeated
pre-holiday pattern is the strongest candidate for future
feature or model work; one-off calendar effects should remain descriptive.


## Scope and limitations

The analysis is diagnostic and observational. Calendar coincidence does not establish causality, and the top-three selection is descriptive rather than a statistical anomaly test. Initial fit rows can confirm recurring demand structure but cannot measure fitted-model error because in-sample predictions were not persisted. Rolling validation predictions cover the occurrence stage only; positive-quantity generalization is therefore assessed from actual historical quantities rather than validation forecasts.